# Classification de Chiffres Manuscrits avec CNNs

## 🎯 Ce Que Vous Allez Apprendre

- Charger et prétraiter le dataset MNIST
- Construire un réseau de neurones complètement connecté (Fully Connected) pour la classification d'images
- Construire et entraîner un réseau de neurones convolutionnel (CNN) pour la classification d'images
- Comprendre l'impact des différentes architectures de réseaux sur les performances
- Fonctionnalités de base de Keras pour la construction et l'entraînement de modèles

## 🛠️ Ce Que Vous Allez Créer

Vous allez créer deux modèles :
1. Un réseau de neurones complètement connecté (couches Dense) pour classifier les chiffres manuscrits du dataset MNIST
2. Un réseau de neurones convolutionnel (CNN) pour classifier les chiffres manuscrits du dataset MNIST, et comparer ses performances avec le premier modèle

In [ ]:
# 1. Charger le dataset MNIST
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

# Charger les données MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Afficher les dimensions des données
print(f"Forme de X_train: {X_train.shape}")  # (60000, 28, 28)
print(f"Forme de y_train: {y_train.shape}")  # (60000,)
print(f"Forme de X_test: {X_test.shape}")    # (10000, 28, 28)
print(f"Forme de y_test: {y_test.shape}")    # (10000,)

# Visualiser quelques exemples
plt.figure(figsize=(10, 2))
for i in range(5):
  plt.subplot(1, 5, i+1)
  plt.imshow(X_train[i], cmap='gray')
  plt.title(f"Label: {y_train[i]}")
  plt.axis('off')
plt.show()

# 2. Prétraiter les données pour un réseau complètement connecté
# Aplatir les images de 28x28 à 784 pixels
X_train_flat = X_train.reshape(60000, 784)
X_test_flat = X_test.reshape(10000, 784)

# Normaliser les valeurs de pixels (0-255 -> 0-1)
X_train_flat = X_train_flat.astype('float32') / 255.0
X_test_flat = X_test_flat.astype('float32') / 255.0

# One-hot encoding des labels
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

print(f"\nDonnées prétraitées pour le modèle Dense:")
print(f"X_train_flat shape: {X_train_flat.shape}")
print(f"y_train_cat shape: {y_train_cat.shape}")

# 3. Construire et entraîner un réseau complètement connecté
model_dense = keras.Sequential([
  layers.Dense(128, activation='relu', input_shape=(784,)),  # Couche cachée 1
  layers.Dropout(0.2),                                        # Régularisation
  layers.Dense(64, activation='relu'),                        # Couche cachée 2
  layers.Dropout(0.2),
  layers.Dense(10, activation='softmax')                      # Couche de sortie (10 classes)
])

# Compiler le modèle
model_dense.compile(
  optimizer='adam',
  loss='categorical_crossentropy',
  metrics=['accuracy']
)

# Afficher le résumé du modèle
print("\nArchitecture du modèle Dense:")
model_dense.summary()

# Entraîner le modèle
print("\nEntraînement du modèle Dense...")
history_dense = model_dense.fit(
  X_train_flat, y_train_cat,
  validation_data=(X_test_flat, y_test_cat),
  epochs=10,
  batch_size=128,
  verbose=1
)

# Évaluer le modèle
test_loss_dense, test_acc_dense = model_dense.evaluate(X_test_flat, y_test_cat, verbose=0)
print(f"\nPrécision du modèle Dense sur le test: {test_acc_dense:.4f}")

# 4. Prétraiter les données pour un CNN
# Redimensionner pour Conv2D: (nb_samples, height, width, channels)
X_train_cnn = X_train.reshape(60000, 28, 28, 1)
X_test_cnn = X_test.reshape(10000, 28, 28, 1)

# Normaliser
X_train_cnn = X_train_cnn.astype('float32') / 255.0
X_test_cnn = X_test_cnn.astype('float32') / 255.0

print(f"\nDonnées prétraitées pour le CNN:")
print(f"X_train_cnn shape: {X_train_cnn.shape}")
print(f"X_test_cnn shape: {X_test_cnn.shape}")

# 5. Construire et entraîner un CNN
model_cnn = keras.Sequential([
  # Premier bloc convolutionnel
  layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
  layers.MaxPooling2D((2, 2)),

  # Deuxième bloc convolutionnel
  layers.Conv2D(64, (3, 3), activation='relu'),
  layers.MaxPooling2D((2, 2)),

  # Troisième bloc convolutionnel
  layers.Conv2D(64, (3, 3), activation='relu'),

  # Aplatir avant les couches denses
  layers.Flatten(),

  # Couches denses
  layers.Dense(64, activation='relu'),
  layers.Dropout(0.2),
  layers.Dense(10, activation='softmax')  # 10 classes (chiffres 0-9)
])

# Compiler le modèle
model_cnn.compile(
  optimizer='adam',
  loss='categorical_crossentropy',
  metrics=['accuracy']
)

# Afficher le résumé
print("\nArchitecture du CNN:")
model_cnn.summary()

# Entraîner le modèle
print("\nEntraînement du CNN...")
history_cnn = model_cnn.fit(
  X_train_cnn, y_train_cat,
  validation_data=(X_test_cnn, y_test_cat),
  epochs=10,
  batch_size=128,
  verbose=1
)

# Évaluer le modèle
test_loss_cnn, test_acc_cnn = model_cnn.evaluate(X_test_cnn, y_test_cat, verbose=0)
print(f"\nPrécision du CNN sur le test: {test_acc_cnn:.4f}")

# 6. Comparer les performances
print("\n" + "="*50)
print("COMPARAISON DES PERFORMANCES")
print("="*50)
print(f"Modèle Dense (Fully Connected):")
print(f"  - Précision sur le test: {test_acc_dense:.4f}")
print(f"  - Perte sur le test: {test_loss_dense:.4f}")
print(f"\nModèle CNN (Convolutionnel):")
print(f"  - Précision sur le test: {test_acc_cnn:.4f}")
print(f"  - Perte sur le test: {test_loss_cnn:.4f}")
print(f"\nAmélioration avec le CNN: {(test_acc_cnn - test_acc_dense):.4f} ({((test_acc_cnn - test_acc_dense) / test_acc_dense * 100):.2f}%)")

# Visualiser les courbes d'apprentissage
plt.figure(figsize=(14, 5))

# Précision
plt.subplot(1, 2, 1)
plt.plot(history_dense.history['accuracy'], label='Dense - Train', linestyle='--')
plt.plot(history_dense.history['val_accuracy'], label='Dense - Val', linestyle='--')
plt.plot(history_cnn.history['accuracy'], label='CNN - Train')
plt.plot(history_cnn.history['val_accuracy'], label='CNN - Val')
plt.xlabel('Époque')
plt.ylabel('Précision')
plt.title('Comparaison de la Précision')
plt.legend()
plt.grid(True)

# Perte
plt.subplot(1, 2, 2)
plt.plot(history_dense.history['loss'], label='Dense - Train', linestyle='--')
plt.plot(history_dense.history['val_loss'], label='Dense - Val', linestyle='--')
plt.plot(history_cnn.history['loss'], label='CNN - Train')
plt.plot(history_cnn.history['val_loss'], label='CNN - Val')
plt.xlabel('Époque')
plt.ylabel('Perte')
plt.title('Comparaison de la Perte')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Visualiser quelques prédictions
predictions = model_cnn.predict(X_test_cnn[:10])
plt.figure(figsize=(15, 3))
for i in range(10):
  plt.subplot(2, 5, i+1)
  plt.imshow(X_test[i], cmap='gray')
  predicted_label = np.argmax(predictions[i])
  true_label = y_test[i]
  color = 'green' if predicted_label == true_label else 'red'
  plt.title(f"Vrai: {true_label}\nPréd: {predicted_label}", color=color)
  plt.axis('off')
plt.suptitle('Prédictions du CNN (Vert=Correct, Rouge=Erreur)')
plt.tight_layout()
plt.show()